In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from gasp import weather_weekly

# Weather weekly demo

All data in this notebook is fake and follows CONTRACTS.md section 4. The module receives DataFrames and never fetches or loads data.

The first fixture covers Saturday December 30, 2023 through Friday January 12, 2024. Each complete week sums to 28, including the week that crosses New Year.

In [2]:
dates = pd.date_range("2023-12-30", "2024-01-12", freq="D")    # Saturday Dec 30 to Friday Jan 12
daily = pd.DataFrame({
    "date": dates,
    "region": "US",
    "measure": "hdd",
    "weighting": "utility_gas",
    "value": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0] * 2,
})

In [3]:
weekly = weather_weekly.to_eia_weeks(daily)
expected_weekly = pd.DataFrame({
    "week_end": pd.to_datetime(["2024-01-05", "2024-01-12"]),
    "region": "US",
    "measure": "hdd",
    "weighting": "utility_gas",
    "value": [28.0, 28.0],
})
pd.testing.assert_frame_equal(weekly, expected_weekly)
assert weekly["week_end"].is_monotonic_increasing

# The exact single-week example from the Done when checklist.
single_week = weather_weekly.to_eia_weeks(
    daily.loc[daily["date"].between("2024-01-06", "2024-01-12")]
)
pd.testing.assert_frame_equal(
    single_week, expected_weekly.iloc[[1]].reset_index(drop=True)
)

# The other week crosses New Year and still totals 28.
cross_year_week = weather_weekly.to_eia_weeks(
    daily.loc[daily["date"].between("2023-12-30", "2024-01-05")]
)
pd.testing.assert_frame_equal(
    cross_year_week, expected_weekly.iloc[[0]].reset_index(drop=True)
)
weekly

,week_end,region,measure,weighting,value
0,2024-01-05,US,hdd,utility_gas,28.0
1,2024-01-12,US,hdd,utility_gas,28.0


## Complete weeks and separate series

A missing day makes a weekly sum artificially low. Drop the entire incomplete week, even when the missing day is in the middle.

Changing a region, measure, or weighting creates a different series. Each series must be summed separately.

In [4]:
# Removing any day makes its whole week incomplete, including a middle day.
for missing_day in [0, 3, 6]:
    partial = weather_weekly.to_eia_weeks(daily.drop(index=missing_day))
    pd.testing.assert_frame_equal(
        partial, expected_weekly.iloc[[1]].reset_index(drop=True)
    )

# An input containing only six days must return an empty table.
six_days = daily.loc[daily["date"].between("2024-01-06", "2024-01-11")]
assert len(six_days) == 6
six_day_result = weather_weekly.to_eia_weeks(six_days)
assert six_day_result.empty
six_day_result

,week_end,region,measure,weighting,value


In [5]:
from IPython.display import display

for column, label, daily_value in [
    ("region", "3", 10.0),
    ("measure", "cdd", 20.0),
    ("weighting", "population", 30.0),
]:
    other_daily = daily.assign(**{column: label, "value": daily_value})
    mixed_daily = pd.concat([daily, other_daily], ignore_index=True)
    separated = weather_weekly.to_eia_weeks(mixed_daily)

    original_series = separated.loc[
        (separated["region"] == "US")
        & (separated["measure"] == "hdd")
        & (separated["weighting"] == "utility_gas")
    ].reset_index(drop=True)
    pd.testing.assert_frame_equal(original_series, expected_weekly)
    assert separated.loc[separated[column] == label, "value"].tolist() == [
        7 * daily_value, 7 * daily_value
    ]
    assert len(separated) == 4
    print(f"{column.capitalize()}: {daily[column].iloc[0]} and {label}")
    display(separated)

print("Weekly grouping checks passed for region, measure, and weighting.")

Region: US and 3


,week_end,region,measure,weighting,value
0,2024-01-05,3,hdd,utility_gas,70.0
1,2024-01-05,US,hdd,utility_gas,28.0
2,2024-01-12,3,hdd,utility_gas,70.0
3,2024-01-12,US,hdd,utility_gas,28.0


Measure: hdd and cdd


,week_end,region,measure,weighting,value
0,2024-01-05,US,cdd,utility_gas,140.0
1,2024-01-05,US,hdd,utility_gas,28.0
2,2024-01-12,US,cdd,utility_gas,140.0
3,2024-01-12,US,hdd,utility_gas,28.0


Weighting: utility_gas and population


,week_end,region,measure,weighting,value
0,2024-01-05,US,hdd,population,210.0
1,2024-01-05,US,hdd,utility_gas,28.0
2,2024-01-12,US,hdd,population,210.0
3,2024-01-12,US,hdd,utility_gas,28.0


Weekly grouping checks passed for region, measure, and weighting.


## Spot-check three normals by hand

Each fake daily value is its calendar year minus 2000. For these midyear weeks, each weekly total is seven times that number. The historical totals form an evenly spaced sequence, so their average is `(first total + last total) / 2`.

| Friday being checked | Prior ten years | First and last historical weekly totals | Normal calculated by hand | Current value | Deviation |
|---|---|---|---|---|---|
| 2010-07-02 | 2000-2009 | 0 and 63 | (0 + 63) / 2 = **31.5** | 70 | 38.5 |
| 2015-06-26 | 2005-2014 | 35 and 98 | (35 + 98) / 2 = **66.5** | 105 | 38.5 |
| 2020-06-26 | 2010-2019 | 70 and 133 | (70 + 133) / 2 = **101.5** | 140 | 38.5 |

The code below compares the function's results with these three independently calculated answers.

For a row in calendar year Y, the normal uses the same ISO week and weather series from exactly Y-10 through Y-1. Every year must be present; an incomplete history gives NaN. Before 2010, this fixture has fewer than ten prior years.

In [6]:
dates = pd.date_range("2000-01-01", "2024-12-31", freq="D")
long_daily = pd.DataFrame({
    "date": dates,
    "region": "US",
    "measure": "hdd",
    "weighting": "utility_gas",
    "value": (dates.year - 2000).astype(float),
})

In [7]:
long_weekly = weather_weekly.to_eia_weeks(long_daily)
long_weekly

,week_end,region,measure,weighting,value
0,2000-01-07,US,hdd,utility_gas,0.0
1,2000-01-14,US,hdd,utility_gas,0.0
2,2000-01-21,US,hdd,utility_gas,0.0
3,2000-01-28,US,hdd,utility_gas,0.0
4,2000-02-04,US,hdd,utility_gas,0.0
...,...,...,...,...,...
1299,2024-11-29,US,hdd,utility_gas,168.0
1300,2024-12-06,US,hdd,utility_gas,168.0
1301,2024-12-13,US,hdd,utility_gas,168.0
1302,2024-12-20,US,hdd,utility_gas,168.0


In [8]:
long_weekly_before = long_weekly.copy(deep=True)
with_normals = weather_weekly.add_normal(long_weekly)

# Adding normals preserves the input, rows, index, and original columns.
pd.testing.assert_frame_equal(long_weekly, long_weekly_before)
pd.testing.assert_frame_equal(with_normals[long_weekly.columns], long_weekly)
assert with_normals.columns.tolist() == long_weekly.columns.tolist() + [
    "normal", "deviation"
]

year = with_normals["week_end"].dt.year
iso_week = with_normals["week_end"].dt.isocalendar().week
assert with_normals.loc[year < 2010, "normal"].isna().all()
assert with_normals.loc[year < 2010, "deviation"].isna().all()

# Weeks 2 through 51 stay inside one calendar year in every historical year.
interior = with_normals.loc[(year >= 2010) & iso_week.between(2, 51)]
assert not interior.empty
assert interior["deviation"].eq(38.5).all()


hand_checks = pd.DataFrame({
    "week_end": pd.to_datetime(["2010-07-02", "2015-06-26", "2020-06-26"]),
    "value": [70.0, 105.0, 140.0],
    "normal": [31.5, 66.5, 101.5],
    "deviation": [38.5, 38.5, 38.5],
})
spot_checks = with_normals.loc[
    with_normals["week_end"].isin(hand_checks["week_end"]),
    hand_checks.columns,
].reset_index(drop=True)
pd.testing.assert_frame_equal(spot_checks, hand_checks)
spot_checks

,week_end,value,normal,deviation
0,2010-07-02,70.0,31.5,38.5
1,2015-06-26,105.0,66.5,38.5
2,2020-06-26,140.0,101.5,38.5


## Missing historical years

For a 2011 row, the required ten-year window is 2001 through 2010. Removing 2005 must give NaN. The available 2000 observation must not replace that missing year.

In [9]:
without_2005 = long_weekly.loc[
    long_weekly["week_end"].dt.year != 2005
].copy()
missing_year_normals = weather_weekly.add_normal(without_2005)
target = pd.Timestamp("2011-07-01")

assert with_normals.loc[
    with_normals["week_end"] == target, "normal"
].notna().all()
missing_target = missing_year_normals.loc[
    missing_year_normals["week_end"] == target
]
assert len(missing_target) == 1
assert missing_target[["normal", "deviation"]].isna().all().all()
missing_target

,week_end,region,measure,weighting,value,normal,deviation
599,2011-07-01,US,hdd,utility_gas,77.0,NaN,NaN


## Leakage test: cut the data off after 2020

First calculate normals using the full history through 2024. Then remove every row after 2020 and calculate normals again. Every remaining row must have exactly the same normal as before, including the same NaN values. This checks that later observations never influence an earlier normal.

In [10]:
through_2020 = long_weekly["week_end"].dt.year <= 2020
cutoff_weekly = long_weekly.loc[through_2020].copy()
assert len(cutoff_weekly) < len(long_weekly)
cutoff_normals = weather_weekly.add_normal(cutoff_weekly)

assert cutoff_normals["normal"].notna().any()
pd.testing.assert_series_equal(
    cutoff_normals["normal"],
    with_normals.loc[through_2020, "normal"],
)
print("Before cutoff: calculated with the full history, showing rows through 2020")
display(with_normals.loc[through_2020])

print("After cutoff: recalculated using only data through 2020")
display(cutoff_normals)

print("Cutoff test passed: every normal through 2020 is unchanged.")

Before cutoff: calculated with the full history, showing rows through 2020


,week_end,region,measure,weighting,value,normal,deviation
0,2000-01-07,US,hdd,utility_gas,0.0,NaN,NaN
1,2000-01-14,US,hdd,utility_gas,0.0,NaN,NaN
2,2000-01-21,US,hdd,utility_gas,0.0,NaN,NaN
3,2000-01-28,US,hdd,utility_gas,0.0,NaN,NaN
4,2000-02-04,US,hdd,utility_gas,0.0,NaN,NaN
...,...,...,...,...,...,...,...
1090,2020-11-27,US,hdd,utility_gas,140.0,101.5,38.5
1091,2020-12-04,US,hdd,utility_gas,140.0,101.5,38.5
1092,2020-12-11,US,hdd,utility_gas,140.0,101.5,38.5
1093,2020-12-18,US,hdd,utility_gas,140.0,101.5,38.5


After cutoff: recalculated using only data through 2020


,week_end,region,measure,weighting,value,normal,deviation
0,2000-01-07,US,hdd,utility_gas,0.0,NaN,NaN
1,2000-01-14,US,hdd,utility_gas,0.0,NaN,NaN
2,2000-01-21,US,hdd,utility_gas,0.0,NaN,NaN
3,2000-01-28,US,hdd,utility_gas,0.0,NaN,NaN
4,2000-02-04,US,hdd,utility_gas,0.0,NaN,NaN
...,...,...,...,...,...,...,...
1090,2020-11-27,US,hdd,utility_gas,140.0,101.5,38.5
1091,2020-12-04,US,hdd,utility_gas,140.0,101.5,38.5
1092,2020-12-11,US,hdd,utility_gas,140.0,101.5,38.5
1093,2020-12-18,US,hdd,utility_gas,140.0,101.5,38.5


Cutoff test passed: every normal through 2020 is unchanged.
